# 📊 So Sánh Dữ Liệu: Clean vs Chuẩn Hóa
## Tỉnh Đắk Nông - Hộ Nghèo/Cận Nghèo 2022

### 🎯 Mục Tiêu Phân Tích
1. **Độ chính xác**: Bao nhiêu % dữ liệu khớp giữa clean và chuẩn?
2. **Các thành phần sai**: Chi tiết từng trường sai ở đâu?
3. **Trực quan hóa sai lệch**: Mức độ, phạm vi, tập trung ở đâu?
4. **Giải thích nguyên nhân**: Tại sao lại sai?

### 📁 Nguồn Dữ Liệu
- **Clean**: `data/clean/*.xlsx` (8 huyện riêng lẻ)
- **Chuẩn**: `Dữ liệu đã chuẩn hóa/File chuẩn hoá_HN_2022/`

### 📊 Output
- `data_comparison_report.png` - Biểu đồ tổng hợp
- `heatmap_district_field.png` - Heatmap chi tiết
- `data_comparison_report.txt` - Báo cáo văn bản
- `data_comparison_detailed.xlsx` - Dữ liệu chi tiết (Excel)


In [18]:
import glob
import os
import pandas as pd
import numpy as np
import matplotlib
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import glob
import re
import warnings
from matplotlib.patches import FancyBboxPatch
from collections import defaultdict

warnings.filterwarnings("ignore")
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style("whitegrid")

output_dir = str(ROOT / "output" / "image")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
# ── 1. LOAD & NORMALIZE ───────────────────────────────────────────────
def load_std(path):
    """Load dữ liệu từ file chuẩn hóa"""
    xl = pd.ExcelFile(path)
    sh = next((s for s in xl.sheet_names if "ng h" in s.lower()), xl.sheet_names[0])
    df = pd.read_excel(path, sheet_name=sh, header=0, dtype=str)
    if str(df.columns[0]).startswith("Unnamed"):
        df = df.iloc[:, 1:]
    return df

# Load dữ liệu
print(" Đang load dữ liệu...")
clean_files = [f for f in glob.glob(str(ROOT / "data" / "clean" / "*.xlsx")) if "Chuan_hoa_Toan_Tinh" not in f]
std_files   = [f for f in glob.glob(str(ROOT / "Dữ liệu đã chuẩn hóa" / "**" / "*.xlsx"), recursive=True) if "MACOSX" not in f]

all_clean = pd.concat([pd.read_excel(f, sheet_name="Du_lieu_chuan_hoa", dtype=str) 
                       for f in clean_files], ignore_index=True)
all_std   = pd.concat([load_std(f) for f in std_files], ignore_index=True)

print(f"  Clean: {len(all_clean)} hàng, {len(all_clean.columns)} cột")
print(f"  Chuẩn: {len(all_std)} hàng, {len(all_std.columns)} cột")

# Hàm normalize
def norm_gender(v):
    v = str(v).strip().upper()
    if v in ["NỮ","NU","FEMALE","2"]: return "Nữ"
    if v in ["NAM","MALE","1"]: return "Nam"
    return "Khác"

def norm_phanloai(v):
    v = str(v).strip().upper()
    if "NEAR_POOR" in v or v == "2": return "Hộ cận nghèo"
    if "POOR" in v or v == "1": return "Hộ nghèo"
    return v

def norm_ketqua(v):
    v = str(v).strip().upper()
    if "CCCD" in v and "TIM" in v: return "Tìm thấy - CCCD"
    if "CMND" in v and "TIM" in v: return "Tìm thấy - CMND"
    if "TIM THAY" in v: return "Tìm thấy - Thông tin"
    if "KHONG TIM" in v or "KHÔNG TÌM" in v: return "Không tìm thấy"
    return "Khác"

def norm_huyen(v):
    v = str(v).strip().upper()
    return re.sub(r"^(HUYỆN |THÀNH PHỐ )", "", v).strip()

def norm_date(v):
    v = str(v).strip()
    m1 = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$", v)
    m2 = re.match(r"^(\d{4})[-/](\d{1,2})[-/](\d{1,2})", v)
    if m1: return f"{m1.group(3)}-{int(m1.group(2)):02d}-{int(m1.group(1)):02d}"
    if m2: return f"{m2.group(1)}-{int(m2.group(2)):02d}-{int(m2.group(3)):02d}"
    return None

# ── 2. MATCH BY ID ────────────────────────────────────────────────────
print(" Đang so sánh CCCD...")
all_clean["id"] = all_clean["SO_CCCD"].str.strip()
all_std["id"]   = all_std["SO_CCCD"].str.strip()

clean_ids = set(all_clean["id"].dropna())
std_ids   = set(all_std["id"].dropna())
both      = clean_ids & std_ids
only_c    = clean_ids - std_ids
only_s    = std_ids   - clean_ids

# Chuẩn bị dữ liệu so sánh
c = all_clean[all_clean["id"].isin(both)].drop_duplicates("id").set_index("id")
s = all_std[all_std["id"].isin(both)].drop_duplicates("id").set_index("id")
idx = c.index.intersection(s.index)
c = c.loc[idx]
s = s.loc[idx]
N = len(idx)

print(f"  Total CCCD (clean): {len(clean_ids):,}")
print(f"  Total CCCD (chuẩn): {len(std_ids):,}")
print(f"  Khớp: {len(both):,} ({len(both)/max(len(clean_ids),1)*100:.2f}%)")
print(f"  Chỉ trong clean: {len(only_c):,}")
print(f"  Chỉ trong chuẩn: {len(only_s):,}")

print(f"\n  Đang phân tích {N:,} bản ghi khớp...")


[1/5] Đang load dữ liệu...
  Clean: 101355 hàng, 18 cột
  Chuẩn: 105507 hàng, 50 cột
[2/5] Đang so sánh CCCD...
  Total CCCD (clean): 86,204
  Total CCCD (chuẩn): 86,091
  Khớp: 80,190 (93.02%)
  Chỉ trong clean: 6,014
  Chỉ trong chuẩn: 5,901

  Đang phân tích 80,190 bản ghi khớp...


In [ ]:
# ── 3. TÍNH TOÁN MATCH RATE TỪNG TRƯỜNG ───────────────────────────────
print("\n Tính toán match rate từng trường...")

def pct(col_c, col_s, fn=None):
    """Tính % khớp giữa 2 cột"""
    a = c[col_c].fillna("").astype(str)
    b = s[col_s].fillna("").astype(str)
    if fn:
        a, b = a.apply(fn), b.apply(fn)
    else:
        a, b = a.str.strip().str.upper(), b.str.strip().str.upper()
    return (a == b).sum() / N * 100

def get_mismatch_details(col_c, col_s, fn=None, max_samples=20):
    """Lấy chi tiết các bản ghi không khớp"""
    a = c[col_c].fillna("").astype(str)
    b = s[col_s].fillna("").astype(str)
    if fn:
        a, b = a.apply(fn), b.apply(fn)
    else:
        a, b = a.str.strip().str.upper(), b.str.strip().str.upper()
    
    mask = (a != b)
    if mask.sum() == 0:
        return pd.DataFrame()
    
    result = pd.DataFrame({
        'clean': c.loc[mask, col_c].head(max_samples).values,
        'chuẩn': s.loc[mask, col_s].head(max_samples).values
    })
    return result

# Tính match rate chính
date_ok = [(norm_date(c.loc[i,"NGAY_SINH"]) == norm_date(s.loc[i,"NGAY_SINH"]))
           for i in idx[:5000] if norm_date(c.loc[i,"NGAY_SINH"]) and norm_date(s.loc[i,"NGAY_SINH"])]
date_pct = sum(date_ok)/len(date_ok)*100 if date_ok else 0

col_match = {
    "Họ tên":      pct("HO_TEN_THANH_VIEN","HO_TEN_THANH_VIEN"),
    "Ngày sinh":   date_pct,
    "Giới tính":   pct("GIOI_TINH","GIOI_TINH", norm_gender),
    "Tỉnh":        pct("TINH","TINH"),
    "Huyện":       pct("HUYEN","HUYEN", norm_huyen),
    "Xã":          pct("XA","XA"),
    "Phân loại HH":pct("PHAN_LOAI_HO","PHAN_LOAI_HO", norm_phanloai),
    "Kết quả":     pct("KET_QUA","KET_QUA", norm_ketqua),
}

# Sắp xếp theo match rate
col_match_sorted = dict(sorted(col_match.items(), key=lambda x: x[1]))
overall_match = sum(col_match.values()) / len(col_match)

print("\n=== TÓM LƯỢC KẾT QUẢ ===")
print(f"Độ khớp TRUNG BÌNH: {overall_match:.2f}%\n")
print("Match rate từng trường (sắp xếp từ thấp đến cao):")
for k, v in col_match_sorted.items():
    bar = "█" * int(v/5) + "░" * (20-int(v/5))
    print(f"  {k:15s} {bar} {v:6.2f}%")



[3/5] Tính toán match rate từng trường...

=== TÓM LƯỢC KẾT QUẢ ===
Độ khớp TRUNG BÌNH: 72.12%

Match rate từng trường (sắp xếp từ thấp đến cao):
  Phân loại HH    ░░░░░░░░░░░░░░░░░░░░   0.00%
  Kết quả         ░░░░░░░░░░░░░░░░░░░░   0.64%
  Huyện           ████████████████░░░░  80.53%
  Họ tên          ███████████████████░  97.42%
  Xã              ███████████████████░  99.40%
  Tỉnh            ███████████████████░  99.53%
  Ngày sinh       ███████████████████░  99.59%
  Giới tính       ███████████████████░  99.86%


In [5]:
# ── 4. PHÂN TÍCH THEO HUYỆN ─────────────────────────────────────────────
print("\n Phân tích match rate theo huyện/tp...")

# Thêm huyện từ clean data
c_with_huyen = c.copy()
c_with_huyen['huyen_norm'] = c_with_huyen['HUYEN'].apply(norm_huyen)

# Tính match rate cho từng trường, từng huyện
district_analysis = {}
for huyen in c_with_huyen['huyen_norm'].unique():
    if pd.isna(huyen) or huyen == "":
        continue
    
    mask = c_with_huyen['huyen_norm'] == huyen
    c_dist = c.loc[mask]
    s_dist = s.loc[mask]
    n_dist = len(c_dist)
    
    if n_dist < 5:
        continue
    
    def pct_dist(col_c, col_s, fn=None):
        if n_dist == 0:
            return 0
        a = c_dist[col_c].fillna("").astype(str)
        b = s_dist[col_s].fillna("").astype(str)
        if fn:
            a, b = a.apply(fn), b.apply(fn)
        else:
            a, b = a.str.strip().str.upper(), b.str.strip().str.upper()
        return (a == b).sum() / n_dist * 100
    
    district_analysis[huyen] = {
        'n': n_dist,
        'avg_match': np.mean([
            pct_dist("HO_TEN_THANH_VIEN","HO_TEN_THANH_VIEN"),
            pct_dist("GIOI_TINH","GIOI_TINH", norm_gender),
            pct_dist("HUYEN","HUYEN", norm_huyen),
            pct_dist("XA","XA"),
        ]),
        'details': {
            'Họ tên': pct_dist("HO_TEN_THANH_VIEN","HO_TEN_THANH_VIEN"),
            'Giới tính': pct_dist("GIOI_TINH","GIOI_TINH", norm_gender),
            'Huyện': pct_dist("HUYEN","HUYEN", norm_huyen),
            'Xã': pct_dist("XA","XA"),
        }
    }

# Sắp xếp theo avg_match
district_df = pd.DataFrame([
    {'Huyện': k, 'N': v['n'], 'Độ khớp (%)': v['avg_match']}
    for k, v in sorted(district_analysis.items(), key=lambda x: x[1]['avg_match'])
])

print("\nMatch rate theo huyện/tp:")
print(district_df.to_string(index=False))

# Lưu để dùng sau
district_analysis_df = district_df.copy()



 Phân tích match rate theo huyện/tp...

Match rate theo huyện/tp:
    Huyện     N  Độ khớp (%)
  ĐẮK MIL   376    48.537234
ĐẮK R'LẤP  4103    74.963441
   CƯ JÚT 11116    74.997751
ĐẮK GLONG 27271    98.005207
GIA NGHĨA    78    99.038462
 KRÔNG NÔ 13234    99.860209
 ĐẮK SONG  8245    99.954518
  TUY ĐỨC 15767    99.995243


In [7]:
# ── 5. CHI TIẾT CÁC SAI LỆP CHÍNH ───────────────────────────────────────
print("\n Phân tích chi tiết sai lệch...")

# Xác định trường có sai lệch cao nhất
worst_field = min(col_match.items(), key=lambda x: x[1])
print(f"\n⚠️  Trường có match rate THẤP NHẤT: {worst_field[0]} ({worst_field[1]:.2f}%)")

# Chi tiết sai lệch
mismatch_summary = {}
for field_name, (clean_col, std_col, norm_fn) in [
    ("Họ tên", ("HO_TEN_THANH_VIEN","HO_TEN_THANH_VIEN", None)),
    ("Ngày sinh", ("NGAY_SINH","NGAY_SINH", norm_date)),
    ("Giới tính", ("GIOI_TINH","GIOI_TINH", norm_gender)),
    ("Huyện", ("HUYEN","HUYEN", norm_huyen)),
    ("Xã", ("XA","XA", None)),
]:
    a = c[clean_col].fillna("").astype(str)
    b = s[std_col].fillna("").astype(str)
    
    if norm_fn:
        a, b = a.apply(norm_fn), b.apply(norm_fn)
    else:
        a, b = a.str.strip().str.upper(), b.str.strip().str.upper()
    
    mask = (a != b)
    n_mismatch = mask.sum()
    pct_mismatch = n_mismatch / N * 100
    
    if n_mismatch > 0:
        mismatch_summary[field_name] = {
            'n_mismatch': n_mismatch,
            'pct_mismatch': pct_mismatch,
            'samples': list(zip(
                c.loc[mask, clean_col].head(5).values,
                s.loc[mask, std_col].head(5).values
            ))
        }
        print(f"\n{field_name}:")
        print(f"  Sai lệch: {n_mismatch:,} / {N:,} ({pct_mismatch:.2f}%)")
        print(f"  Mẫu sai lệch:")
        for i, (v_clean, v_std) in enumerate(mismatch_summary[field_name]['samples'], 1):
            print(f"    {i}. Clean: '{v_clean}' → Chuẩn: '{v_std}'")



 Phân tích chi tiết sai lệch...

⚠️  Trường có match rate THẤP NHẤT: Phân loại HH (0.00%)

Họ tên:
  Sai lệch: 2,066 / 80,190 (2.58%)
  Mẫu sai lệch:
    1. Clean: 'ĐIÊU THỊ KHÁNH' → Chuẩn: 'ĐIÊU THỊ KHÁNH
 HUYỀN'
    2. Clean: 'H' → Chuẩn: 'H,JU LY KIỀU'
    3. Clean: 'NÔNG THẾ HUẤN (CHUYỂN ĐI GIA LAI' → Chuẩn: 'NÔNG THẾ HUẤN (CHUYỂN ĐI GIA LAI, CẢI CHÍNH THÀNH LA VĂN HUẤN)'
    4. Clean: 'VÕ THANH TRÚC' → Chuẩn: 'VÕ THÀNH PHÚ'
    5. Clean: 'H' MĂNG' → Chuẩn: 'H' MĂNG (B)'

Ngày sinh:
  Sai lệch: 10,134 / 80,190 (12.64%)
  Mẫu sai lệch:
    1. Clean: '01-01-1939' → Chuẩn: '22/04/1905'
    2. Clean: '27-06-1980' → Chuẩn: '02/06/1905'
    3. Clean: '10-11-1981' → Chuẩn: '03/06/1905'
    4. Clean: '09-12-2011' → Chuẩn: '03/07/1905'
    5. Clean: '27-12-2013' → Chuẩn: '05/07/1905'

Giới tính:
  Sai lệch: 112 / 80,190 (0.14%)
  Mẫu sai lệch:
    1. Clean: 'Nữ' → Chuẩn: 'MALE'
    2. Clean: 'Nữ' → Chuẩn: '1'
    3. Clean: 'Nữ' → Chuẩn: '1'
    4. Clean: 'Nữ' → Chuẩn: '1'
    5. Clean: 'Na

In [19]:
# ──  TRỰC QUAN HÓA 1: MATCH RATE TỪNG TRƯỜNG ──────────────────────────
print("\n Tạo biểu đồ...")

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)

# Biểu đồ cột - Match rate từng trường
ax1 = fig.add_subplot(gs[0, :])
fields = list(col_match.keys())
values = list(col_match.values())
colors = ['🔴' if v < 80 else '🟡' if v < 90 else '🟢' for v in values]
bars = ax1.barh(fields, values, color=['#d62728' if v < 80 else '#ff7f0e' if v < 90 else '#2ca02c' for v in values])
ax1.set_xlabel('Match Rate (%)', fontsize=12, fontweight='bold')
ax1.set_title('Match Rate Từng Trường (So Sánh Clean vs Chuẩn)', fontsize=14, fontweight='bold')
ax1.set_xlim([0, 105])
for i, (bar, val) in enumerate(zip(bars, values)):
    ax1.text(val + 1, i, f'{val:.2f}%', va='center', fontsize=10, fontweight='bold')
ax1.axvline(overall_match, color='blue', linestyle='--', linewidth=2, label=f'Trung bình: {overall_match:.2f}%')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Biểu đồ cột - Match rate theo huyện
if len(district_analysis_df) > 0:
    ax2 = fig.add_subplot(gs[1, :])
    district_analysis_df_plot = district_analysis_df.sort_values('Độ khớp (%)')
    bars2 = ax2.barh(district_analysis_df_plot['Huyện'], district_analysis_df_plot['Độ khớp (%)'],
                     color=['#d62728' if v < 80 else '#ff7f0e' if v < 90 else '#2ca02c' 
                           for v in district_analysis_df_plot['Độ khớp (%)']])
    ax2.set_xlabel('Match Rate (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Match Rate Theo Huyện (N = số bản ghi)', fontsize=14, fontweight='bold')
    ax2.set_xlim([0, 105])
    for i, (idx, row) in enumerate(district_analysis_df_plot.iterrows()):
        ax2.text(row['Độ khớp (%)'] + 1, i, f"{row['Độ khớp (%)']:.1f}% (N={row['N']})", 
                va='center', fontsize=9)
    ax2.grid(axis='x', alpha=0.3)

# Pie chart - Tỷ lệ khớp/không khớp
ax3 = fig.add_subplot(gs[2, 0])
matched = len(both)
unmatched = len(clean_ids) - len(both)
ax3.pie([matched, unmatched], labels=['Khớp CCCD', 'Không khớp CCCD'], 
        autopct='%1.1f%%', startangle=90, colors=['#2ca02c', '#d62728'])
ax3.set_title('Tỷ lệ Khớp CCCD\n(Clean: {}, Chuẩn: {})'.format(
    len(clean_ids), len(std_ids)), fontsize=12, fontweight='bold')

# Ma trận sai lệch - trường
ax4 = fig.add_subplot(gs[2, 1])
fields_cam = ["Họ tên", "Ngày sinh", "Giới tính", "Tỉnh", "Huyện", "Xã", "Phân loại HH", "Kết quả"]
error_percentages = [2.58, 12.64, 0.14, 0.47, 19.47, 0.60, 100.0, 99.36]

# Sử dụng ax4.bar thay vì plt.bar để ép dữ liệu vào đúng ô lưới hành chính
bars4 = ax4.bar(fields_cam, error_percentages, color="#ff7f0e", width=0.55)
ax4.set_title("Tỷ Lệ Sai Lệch Nội Dung Theo Trường (%)", fontsize=12, fontweight='bold', pad=10)
ax4.set_ylabel("Sai Lệch (%)", fontsize=10)
ax4.set_ylim([0, 115])
ax4.set_xticklabels(fields_cam, rotation=35, ha="right", fontsize=9)
ax4.grid(axis='y', alpha=0.3)

# Điền nhãn số chính xác lên đầu từng cột cam
for bar in bars4:
    height = bar.get_height()
    ax4.text(
        bar.get_x() + bar.get_width()/2.0, 
        height + 1.5, 
        f'{height:.1f}%', 
        ha='center', 
        va='bottom', 
        fontsize=9, 
        fontweight='bold'
    )


plt.savefig(os.path.join(output_dir, 'data_comparison_report.png'), dpi=300, bbox_inches='tight')
print("✓ Biểu đồ đã lưu: data_comparison_report.png")
plt.close()



 Tạo biểu đồ...
✓ Biểu đồ đã lưu: data_comparison_report.png


In [12]:
# ── 7. PHÂN TÍCH PATTERN SỐ LIỆU SAI LỆCH ───────────────────────────────

print("\n" + "="*70)
print("PHÂN TÍCH PATTERN & NGUYÊN NHÂN CHUYÊN SÂU")
print("="*70)

# Lấy danh sách các Index thực tế của mảng giao nhau để chạy vòng lặp
shared_indices = c.index

# 📊 PATTERN 1: Các loại lỗi phổ biến theo thuộc tính
print("\n PATTERN 1: Phân tích bản chất lỗi của từng thuộc tính")
print("-" * 70)

# --- 1. PHÂN TÍCH HỌ TÊN ---
c_name = c["HO_TEN_THANH_VIEN"].fillna("").astype(str).str.strip()
s_name = s["HO_TEN_THANH_VIEN"].fillna("").astype(str).str.strip()
name_mask = c_name.str.upper() != s_name.str.upper()  # So sánh không phân biệt hoa thường trước

# Tính toán chính xác trên toàn bộ các dòng lỗi Họ tên
n_name_errors = name_mask.sum()
if n_name_errors > 0:
    print(f"\n   ★ HỌ TÊN ({n_name_errors:,} sai lệch - {n_name_errors/N*100:.2f}%):")
    
    # Lấy ra tối đa 100 dòng lỗi để thống kê pattern mẫu
    error_indices = c[name_mask].index[:100]
    samples_name = [(c_name.loc[i], s_name.loc[i]) for i in error_indices]
    
    # Thống kê nguyên nhân cụ thể
    spacing_issues = sum(1 for vc, vs in samples_name if vc.replace(" ","") == vs.replace(" ",""))
    case_issues = sum(1 for vc, vs in samples_name if vc.upper() == vs.upper() and vc != vs)
    
    print(f"      - Do lệch định dạng chữ HOA / chữ thường: ~{case_issues/len(samples_name)*100:.1f}%")
    print(f"      - Do thừa/thiếu ký tự khoảng trắng (Space): ~{spacing_issues/len(samples_name)*100:.1f}%")
    print(f"      - Do lệch ký tự nội dung (Tên đệm, cải chính hộ tịch): ~{(len(samples_name) - spacing_issues - case_issues)/len(samples_name)*100:.1f}%")
    
    # In ví dụ đại diện dòng đầu tiên bị lỗi
    first_err_idx = c[name_mask].index[0]
    print(f"      VD mẫu: Clean='{c.loc[first_err_idx, 'HO_TEN_THANH_VIEN']}' ➔ Chuẩn='{s.loc[first_err_idx, 'HO_TEN_THANH_VIEN']}'")

# --- 2. PHÂN TÍCH NGÀY SINH ---
# Chuẩn hóa tạm thời để đếm số ca lệch thực tế
c_date_norm = c["NGAY_SINH"].apply(norm_date)
s_date_norm = s["NGAY_SINH"].apply(norm_date)
date_mask = c_date_norm != s_date_norm
n_date_errors = date_mask.sum()

if n_date_errors > 0:
    print(f"\n   ★ NGÀY SINH ({n_date_errors:,} sai lệch - {n_date_errors/N*100:.2f}%):")
    
    # Lấy mẫu xem có phải do lỗi hệ thống năm 1905 không
    s_date_raw = s["NGAY_SINH"].fillna("").astype(str)
    y1905_issues = sum(1 for i in c[date_mask].index[:100] if "1905" in s_date_raw.loc[i])
    
    if y1905_issues > 0:
        print(f"      - Lỗi hệ thống: Tập CHUẨN bị nhảy đồng loạt về năm gốc 1905 (Lỗi định dạng Excel): ~{y1905_issues:.1f}% trong các mẫu kiểm toán.")
    print(f"      - Lỗi định dạng thô: Vênh cấu trúc chuỗi ghi nhận (DD/MM/YYYY vs YYYY-MM-DD)")
    
    first_date_err = c[date_mask].index[0]
    print(f"      VD mẫu: Clean='{c.loc[first_date_err, 'NGAY_SINH']}' ➔ Chuẩn='{s.loc[first_date_err, 'NGAY_SINH']}'")

# --- 3. PHÂN TÍCH PHÂN LOẠI HỘ ---
if "PHAN_LOAI_HO" in c.columns and "PHAN_LOAI_HO" in s.columns:
    c_phan = c["PHAN_LOAI_HO"].fillna("").astype(str).str.strip().str.upper()
    s_phan = s["PHAN_LOAI_HO"].fillna("").astype(str).str.strip().str.upper()
    phan_mask = c_phan != s_phan
    n_phan_errors = phan_mask.sum()
    
    print(f"\n   ★ PHÂN LOẠI HỘ ({n_phan_errors:,} sai lệch - {n_phan_errors/N*100:.2f}%):")
    if n_phan_errors > 0:
        print(f"      - Nguyên nhân: Lệch pha mã danh mục hệ thống (Chuỗi văn bản tiếng Việt đầy đủ vs Mã viết tắt database)")
        print(f"      VD mẫu các dòng lệch:")
        for idx_id in c[phan_mask].index[:3]:
            print(f"        CCCD {idx_id}: Clean='{c.loc[idx_id, 'PHAN_LOAI_HO']}' ➔ Chuẩn='{s.loc[idx_id, 'PHAN_LOAI_HO']}'")

# 📍 PATTERN 2: Phân hóa mức độ sai lệch theo địa bàn
print(f"\n📍 PATTERN 2: Tập trung sai lệch theo địa giới hành chính")
print("-" * 70)
if len(district_analysis_df) > 0:
    worst_district = district_analysis_df.loc[district_analysis_df['Độ khớp (%)'].idxmin()]
    best_district = district_analysis_df.loc[district_analysis_df['Độ khớp (%)'].idxmax()]
    print(f"   ➔ Huyện đạt độ khớp CAO NHẤT: {best_district['Huyện']} ({best_district['Độ khớp (%)']:.2f}%, N={best_district['N']:,} hộ)")
    print(f"   ➔ Huyện tồn tại sai lệch LỚN NHẤT: {worst_district['Huyện']} ({worst_district['Độ khớp (%)']:.2f}%, N={worst_district['N']:,} hộ)")
    print(f"   ➔ Nhận xét: Sai lệch có tính chất phân cụm hành vi rõ rệt theo từng tệp dữ liệu của từng địa phương nhập liệu.")

# 📈 PATTERN 3: Phân tích độ lệch ổn định của thuật toán
print(f"\n📈 PATTERN 3: Phân tích biến độ lệch (Độ ổn định dữ liệu)")
print("-" * 70)
# Lấy danh sách điểm số từ dict kết quả col_match
match_scores = list(col_match.values())
std_dev = np.std(match_scores)
print(f"   ➔ Độ lệch chuẩn (Standard Deviation) của Match Rate giữa các trường: {std_dev:.2f}%")
if std_dev > 20:
    print(f"   ➔ Đánh giá cấu trúc: Độ lệch chuẩn RẤT CAO (>20%). Dữ liệu sai lệch KHÔNG ĐỀU.")
    print(f"   ➔ Bản chất: Thuật toán tiền xử lý text hành chính làm việc rất tốt, điểm số sụt giảm hoàn toàn cục bộ tại các trường danh mục nghiệp vụ đặc thù.")
else:
    print(f"   ➔ Đánh giá cấu trúc: Độ lệch chuẩn thấp, dữ liệu có độ khớp đồng đều giữa các trường.")

print("\n" + "="*70)


PHÂN TÍCH PATTERN & NGUYÊN NHÂN CHUYÊN SÂU

 PATTERN 1: Phân tích bản chất lỗi của từng thuộc tính
----------------------------------------------------------------------

   ★ HỌ TÊN (2,066 sai lệch - 2.58%):
      - Do lệch định dạng chữ HOA / chữ thường: ~0.0%
      - Do thừa/thiếu ký tự khoảng trắng (Space): ~12.0%
      - Do lệch ký tự nội dung (Tên đệm, cải chính hộ tịch): ~88.0%
      VD mẫu: Clean='ĐIÊU THỊ KHÁNH' ➔ Chuẩn='ĐIÊU THỊ KHÁNH
 HUYỀN'

   ★ NGÀY SINH (10,134 sai lệch - 12.64%):
      - Lỗi hệ thống: Tập CHUẨN bị nhảy đồng loạt về năm gốc 1905 (Lỗi định dạng Excel): ~12.0% trong các mẫu kiểm toán.
      - Lỗi định dạng thô: Vênh cấu trúc chuỗi ghi nhận (DD/MM/YYYY vs YYYY-MM-DD)
      VD mẫu: Clean='01-01-1939' ➔ Chuẩn='22/04/1905'

   ★ PHÂN LOẠI HỘ (80,190 sai lệch - 100.00%):
      - Nguyên nhân: Lệch pha mã danh mục hệ thống (Chuỗi văn bản tiếng Việt đầy đủ vs Mã viết tắt database)
      VD mẫu các dòng lệch:
        CCCD 054139002355: Clean='Hộ nghèo' ➔ Chuẩn='na

In [20]:
# ── 8. TRỰC QUAN HÓA 2: HEATMAP SÂU ─────────────────────────────────────

# Tạo heatmap chi tiết
fig, ax = plt.subplots(figsize=(12, 6))

# Chuẩn bị dữ liệu
district_list = district_analysis_df['Huyện'].tolist() if len(district_analysis_df) > 0 else []
fields_to_plot = ['Họ tên', 'Giới tính', 'Huyện', 'Xã']

if len(district_list) > 0 and len(fields_to_plot) > 0:
    heatmap_data = []
    for huyen in district_list:
        if huyen in district_analysis:
            row = [district_analysis[huyen]['details'].get(f, 100) for f in fields_to_plot]
            heatmap_data.append(row)
    
    if heatmap_data:
        heatmap_array = np.array(heatmap_data)
        sns.heatmap(heatmap_array, annot=True, fmt='.1f', cmap='RdYlGn', 
                    xticklabels=fields_to_plot, yticklabels=district_list,
                    cbar_kws={'label': 'Match Rate (%)'}, ax=ax, vmin=0, vmax=100)
        ax.set_title('Heatmap Match Rate: Trường x Huyện', fontsize=14, fontweight='bold')
        ax.set_xlabel('Trường Dữ Liệu', fontsize=12, fontweight='bold')
        ax.set_ylabel('Huyện', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'heatmap_district_field.png'), dpi=300, bbox_inches='tight')
        print("✓ Heatmap đã lưu: heatmap_district_field.png")
        plt.close()

print("\n✓ Tất cả biểu đồ đã tạo thành công!")


✓ Heatmap đã lưu: heatmap_district_field.png

✓ Tất cả biểu đồ đã tạo thành công!


In [25]:
# ── 9. BÁO CÁO TÓM LƯỢC & KẾT LUẬN ───────────────────────────────────────

report_dir = str(ROOT / "reports")

if not os.path.exists(report_dir):
    os.makedirs(report_dir)

# Gộp thư mục và tên file thành đường dẫn đầy đủ
report_path = os.path.join(report_dir, 'data_comparison_report.txt')

report = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    BÁNG CÁO SO SÁNH DỮ LIỆU - TÓMLƯỢC                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 I. THỐNG KÊ CHUNG
───────────────────────────────────────────────────────────────────────────────
  • Tổng CCCD clean:           {len(clean_ids):>15,}
  • Tổng CCCD chuẩn:           {len(std_ids):>15,}
  • Khớp CCCD:                 {len(both):>15,} ({len(both)/max(len(clean_ids),1)*100:>6.2f}%)
  • Chỉ trong clean:           {len(only_c):>15,}
  • Chỉ trong chuẩn:           {len(only_s):>15,}
  • Số bản ghi so sánh:        {N:>15,}

✅ II. ĐỘ CHÍNH XÁC TỪNG TRƯỜNG
───────────────────────────────────────────────────────────────────────────────
"""

for field, match_pct in col_match_sorted.items():
    status = "🔴" if match_pct < 70 else "🟡" if match_pct < 85 else "🟢"
    report += f"  {status} {field:20s}: {match_pct:6.2f}%  (Sai: {100-match_pct:5.2f}%)\n"

report += f"""
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📈 ĐỘ CHÍNH XÁC TRUNG BÌNH:   {overall_match:>6.2f}%
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📍 III. PHÂN TÍCH THEO ĐỊA PHƯƠNG
───────────────────────────────────────────────────────────────────────────────
"""

if len(district_analysis_df) > 0:
    worst_dist = district_analysis_df.loc[district_analysis_df['Độ khớp (%)'].idxmin()]
    best_dist = district_analysis_df.loc[district_analysis_df['Độ khớp (%)'].idxmax()]
    
    report += f"""  Huyện tốt nhất:    {best_dist['Huyện']:25s} {best_dist['Độ khớp (%)']:6.2f}%  (N={best_dist['N']})
  Huyện tệ nhất:     {worst_dist['Huyện']:25s} {worst_dist['Độ khớp (%)']:6.2f}%  (N={worst_dist['N']})
  Số huyện phân tích: {len(district_analysis_df):3d}
  Sai lệch giữa huyện: {best_dist['Độ khớp (%)'] - worst_dist['Độ khớp (%)']:6.2f}%
"""

report += f"""
🔍 IV. NGUYÊN NHÂN CHÍNH CỦA SAI LỆCH
───────────────────────────────────────────────────────────────────────────────
  1. ❌ ĐỊNH DẠNG KHÔNG NHẤT QUÁN
     • Họ tên: Khoảng trắng, chữ hoa/thường không kiểm soát
     • Ngày sinh: DD/MM/YYYY vs YYYY-MM-DD không đồng nhất
     • Phân loại hộ: Mã số (1/2) vs tên Tiếng Việt

  2. ❌ CHẤT LƯỢNG NHẬP LIỆU
     • Một số huyện có sai lệch cao → quy trình chuẩn hóa khác nhau
     • Có thể do thời gian nhập liệu khác nhau

  3. ❌ CÓ THỂ LỖI CÀN CẢN:
     • Dữ liệu chỉ khớp {len(both):,}/{len(clean_ids):,} → có {len(only_c):,} bản ghi "mất"

💡 V. KHUYẾN NGHỊ
───────────────────────────────────────────────────────────────────────────────
  ✓ Chuẩn hóa ĐỊNH DẠNG: 
    - Họ tên: UPPER() + trim()
    - Ngày sinh: Quy chuẩn về YYYY-MM-DD
    - Phân loại: Sử dụng tên Tiếng Việt thay mã số

  ✓ KIỂM SOÁT CHẤT LƯỢNG:
    - Xác định huyện có sai lệch cao
    - Huấn luyện lại nhân viên nhập liệu

  ✓ QUI TRÌNH:
    - Xây dựng validation rules cho từng trường
    - Tự động kiểm tra trước khi lưu

════════════════════════════════════════════════════════════════════════════════
Ngày sinh tạo báo cáo: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
════════════════════════════════════════════════════════════════════════════════
"""

print(report)

# Lưu báo cáo vào file
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n✓ Báo cáo đã lưu: {report_path}")
print("✓ Hoàn tất phân tích!")



╔══════════════════════════════════════════════════════════════════════════════╗
║                    BÁNG CÁO SO SÁNH DỮ LIỆU - TÓMLƯỢC                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 I. THỐNG KÊ CHUNG
───────────────────────────────────────────────────────────────────────────────
  • Tổng CCCD clean:                    86,204
  • Tổng CCCD chuẩn:                    86,091
  • Khớp CCCD:                          80,190 ( 93.02%)
  • Chỉ trong clean:                     6,014
  • Chỉ trong chuẩn:                     5,901
  • Số bản ghi so sánh:                 80,190

✅ II. ĐỘ CHÍNH XÁC TỪNG TRƯỜNG
───────────────────────────────────────────────────────────────────────────────
  🔴 Phân loại HH        :   0.00%  (Sai: 100.00%)
  🔴 Kết quả             :   0.64%  (Sai: 99.36%)
  🟡 Huyện               :  80.53%  (Sai: 19.47%)
  🟢 Họ tên              :  97.42%  (Sai:  2.58%)
  🟢 Xã                  :  99.40%  (Sai:  0.60%)
  🟢 T

In [22]:
# ── 10. XUẤT DỮ LIỆU PHÂN TÍCH CHI TIẾT (EXCEL) ──────────────────────────

print("\n" + "="*70)
print("XUẤT DỮ LIỆU CHI TIẾT")
print("="*70)

# Định nghĩa tên thư mục bạn muốn chứa file Excel
excel_dir = str(ROOT / "data" / "comparition")

# Tự động tạo thư mục này nếu trên máy chưa có
if not os.path.exists(excel_dir):
    os.makedirs(excel_dir)

# Gộp thư mục và tên file thành đường dẫn đầy đủ
excel_path = os.path.join(excel_dir, 'data_comparison_detailed.xlsx')

# Tạo file Excel với nhiều sheet
try:
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        
        # Sheet 1: Tóm lược
        summary_data = {
            'Chỉ số': [
                'Tổng CCCD (clean)',
                'Tổng CCCD (chuẩn)',
                'Khớp CCCD',
                'Chỉ trong clean',
                'Chỉ trong chuẩn',
                'Số bản ghi so sánh',
                'Độ chính xác trung bình'
            ],
            'Giá trị': [
                f"{len(clean_ids):,}",
                f"{len(std_ids):,}",
                f"{len(both):,}",
                f"{len(only_c):,}",
                f"{len(only_s):,}",
                f"{N:,}",
                f"{overall_match:.2f}%"
            ]
        }
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_excel(writer, sheet_name='Tóm lược', index=False)
        
        # Sheet 2: Match rate từng trường
        field_df = pd.DataFrame([
            {'Trường': k, 'Match (%)': f"{v:.2f}", 'Sai (%)': f"{100-v:.2f}"}
            for k, v in col_match.items()
        ]).sort_values('Match (%)')
        field_df.to_excel(writer, sheet_name='Match Rate Trường', index=False)
        
        # Sheet 3: Match rate theo huyện
        if len(district_analysis_df) > 0:
            district_analysis_df.sort_values('Độ khớp (%)', ascending=False).to_excel(
                writer, sheet_name='Match Rate Huyện', index=False
            )
        
        # Sheet 4: Chi tiết sai lệch
        mismatch_detail_list = []
        for field_name, (clean_col, std_col, norm_fn) in [
            ("Họ tên", ("HO_TEN_THANH_VIEN","HO_TEN_THANH_VIEN", None)),
            ("Ngày sinh", ("NGAY_SINH","NGAY_SINH", norm_date)),
            ("Giới tính", ("GIOI_TINH","GIOI_TINH", norm_gender)),
            ("Huyện", ("HUYEN","HUYEN", norm_huyen)),
            ("Xã", ("XA","XA", None)),
        ]:
            a = c[clean_col].fillna("").astype(str)
            b = s[std_col].fillna("").astype(str)
            
            if norm_fn:
                a, b = a.apply(norm_fn), b.apply(norm_fn)
            else:
                a, b = a.str.strip().str.upper(), b.str.strip().str.upper()
            
            mask = (a != b)
            #for idx_val in c[mask].index[:100]: 
            for idx_val in c[mask].index:
                mismatch_detail_list.append({
                    'Trường lỗi': field_name,
                    'CCCD': idx_val,
                    'Huyện': c.loc[idx_val, 'HUYEN'] if 'HUYEN' in c.columns else "N/A",
                    'Xã': c.loc[idx_val, 'XA'] if 'XA' in c.columns else "N/A",
                    'Giá trị Clean': c.loc[idx_val, clean_col],
                    'Giá trị Chuẩn ': s.loc[idx_val, std_col],
                    'Loại sai': 'Format' if field_name in ['Ngày sinh', 'Giới tính', 'Huyện'] else 'Dữ liệu'
                })
        
        if mismatch_detail_list:
            mismatch_df = pd.DataFrame(mismatch_detail_list)
            mismatch_df.to_excel(writer, sheet_name='Chi tiết Sai lệch', index=False)
        
        print(f"✓ File Excel đã tạo: {excel_path}")
        print(f"  Số sheet: 4 (Tóm lược, Match Rate Trường, Match Rate Huyện, Chi tiết Sai lệch)")
        
except Exception as e:
    print(f"⚠ Không thể xuất Excel (thiếu openpyxl): {e}")
    print("  Cài đặt: pip install openpyxl")

print("\n" + "="*70)
print("✓ HOÀN TẤT! Các file output:")
print("  1. data_comparison_report.png       - Biểu đồ tổng hợp")
print("  2. heatmap_district_field.png       - Heatmap chi tiết")
print("  3. data_comparison_report.txt       - Báo cáo text")
print("  4. data_comparison_detailed.xlsx    - Dữ liệu chi tiết (Excel)")
print("="*70)



XUẤT DỮ LIỆU CHI TIẾT
✓ File Excel đã tạo: C:\NguyenHoangThanhTam\data\comparition\data_comparison_detailed.xlsx
  Số sheet: 4 (Tóm lược, Match Rate Trường, Match Rate Huyện, Chi tiết Sai lệch)

✓ HOÀN TẤT! Các file output:
  1. data_comparison_report.png       - Biểu đồ tổng hợp
  2. heatmap_district_field.png       - Heatmap chi tiết
  3. data_comparison_report.txt       - Báo cáo text
  4. data_comparison_detailed.xlsx    - Dữ liệu chi tiết (Excel)
